# Milestone 4 lab, notebook 7 of 7: The production version

## Where we are

| Notebook | What you do |
|---|---|
| 1. Load first, transform later | Python stops transforming. Compare ELT with milestone 1's ETL |
| 2. Your first model | Write one SELECT in a file, and let dbt build it |
| 3. Clean, join, enrich | One row per order again, then two joins and their traps |
| 4. A loop or a set | Answer one question row by row, then with one statement |
| 5. Layers and the incremental model | Three layers, and a report that rebuilds only its newest days |
| 6. Tests on the tables | Break the data on purpose, and watch the pipeline refuse it |
| **7. The production version** (you are here) | One morning as ELT: Python loads, dbt transforms and tests |

You have run the two halves separately: the Python load, and `dbt build`. The production pipeline runs them one after the other, and explains what changed.

In [1]:
import inspect

from lab_helpers import api, lab_warehouse, run_pipeline, set_morning, show_doc, table

from daily_orders import pipeline, transform
from daily_orders.report import format_answer

## Step 1: The parts

In [2]:
show_doc("LLD.md", "1. Files")

**From `docs/LLD.md`**

> ## 1. Files
> 
> Everything is inside `src/m4_models_and_tests/`. Nothing imports from another milestone.
> 
> ```
> docs/                       BRD.md, HLD.md, LLD.md
> notebooks/                  the lab: seven notebooks, run in order
> daily_orders/               Python: extract and load, and running dbt
>   extract.py                changed orders (with retry), and deletes from the change feed
>   warehouse.py              the raw and ops tables
>   load.py                   land raw; save the watermark
>   transform.py              run dbt, and sum up what it did
>   report.py                 read the report; explain what changed
>   pipeline.py               one morning: extract, load, then dbt build
>   flow_data.py              builds the numbers shown on the pages
> dbt_project/                SQL: every transformation, and every table test
>   dbt_project.yml           the layers, and their settings
>   profiles.yml              how dbt reaches the warehouse
>   models/sources.yml        the raw tables, and the freshness check
>   models/staging/           stg_orders, stg_order_items, stg_order_deletes
>   models/core/              orders, order_items
>   models/marts/             daily_orders
>   seeds/cities.csv          Arden's three cities
>   tests/                    three tests written as SQL
>   macros/                   a schema-name rule, and one test of our own
> tests/                      pytest: the pipeline, and that the table tests fail when they should
> ```

The folder has two halves now. `daily_orders/` is Python: it talks to the order system. `dbt_project/` is SQL: it holds every rule about what the data means.

Here is one morning. It is two calls.

In [3]:
print(inspect.getsource(pipeline.run))

def run(
    client: httpx.Client,
    con: psycopg.Connection,
    sleep: Callable[[float], None] = time.sleep,
    lookback_days: int = LOOKBACK_DAYS,
) -> RunResult:
    """Run the pipeline once, for the morning on the order system's clock."""
    loaded = extract_and_load(client, con, sleep, lookback_days)
    return transform_and_test(con, loaded)



In [4]:
print(inspect.getsource(pipeline.extract_and_load))

def extract_and_load(
    client: httpx.Client,
    con: psycopg.Connection,
    sleep: Callable[[float], None] = time.sleep,
    lookback_days: int = LOOKBACK_DAYS,
) -> dict:
    """E and L: read what changed, and land it raw. One transaction. Returns facts about the run."""
    run_date = read_run_date(client)
    watermark = read_watermark(con)
    since = watermark - timedelta(days=lookback_days) if watermark else None
    extract = fetch_changed_orders(client, to_api_text(since), sleep)
    deletes, feed_position = fetch_deletes(client, read_feed_position(con))
    # The last morning whose report passed every test.
    previous = con.execute(
        "SELECT max(run_date) FROM ops.load_runs WHERE tests_failed = 0"
    ).fetchone()[0]
    with con.transaction():
        run_id = next_run_id(con, run_date)
        now = datetime.now(UTC)
        land(con, extract.orders, extract.pages, run_id, now)
        land_deletes(con, deletes, run_id, now)
        newest = max((order["updated

Compare this with milestone 3's version. The MERGE, the soft delete and the rebuild of the report are gone from Python. What is left is: read, land, save the watermark.

## Step 2: Run it

In [5]:
con = lab_warehouse(fresh=True)
result = run_pipeline(con, "2026-06-08", "2026-06-09")

print(f"Run {result.run_id}: asked for orders changed since {result.asked_since}.")
print(f"Landed {result.rows_landed} rows. dbt built {result.models_built} models and passed {result.tests_passed} tests on the tables.\n")
print(format_answer(result.answer))

Run 2026-06-09_1: asked for orders changed since 2026-06-06T03:48:53Z.
Landed 102 rows. dbt built 7 models and passed 31 tests on the tables.

Arden daily orders. Report for the morning of Tue 2026-06-09

  Yesterday   Mon 2026-06-08    25 orders      $591.75
  Day before  Sun 2026-06-07    30 orders      $807.11
  Change                         -5 orders     -$215.36

  Earlier days that changed in this run:
    Fri 2026-06-05   +0 orders   -$5.78
      because order 1130: refund of $5.78


The same answer as milestone 3, with a reason for the change to Friday. This time **31 tests on the tables** passed before the manager saw it.

From a terminal, in this milestone's folder:

```
uv run python -m daily_orders                 # one morning
uv run python -m daily_orders --full-refresh  # rebuild every day of the report
uv run python -m daily_orders --freshness     # is the raw data recent?
```

## Step 3: The log now knows about the tests

In [6]:
table(con, "SELECT run_id, rows_landed, deletes_landed, api_calls, models_built, tests_passed, tests_failed FROM ops.load_runs ORDER BY run_id")

,run_id,rows_landed,deletes_landed,api_calls,models_built,tests_passed,tests_failed
0,2026-06-08_1,226,0,10,7,31,0
1,2026-06-09_1,102,0,5,7,31,0


## Step 4: A bad morning

Make the order system send a bad row, and run the whole pipeline.

In [7]:
from psycopg.types.json import Jsonb

payload = con.execute("SELECT payload FROM raw.orders_landing WHERE (payload->>'order_id')::bigint = 1130 ORDER BY landing_id DESC LIMIT 1").fetchone()[0]
payload.update(status="teleported", updated_at="2026-06-30T00:00:00Z")
con.execute("INSERT INTO raw.orders_landing (payload, run_id, page_no, loaded_at) VALUES (%s, 'bad_data', 1, now())", [Jsonb(payload)])

set_morning("2026-06-10")
try:
    pipeline.run(api, con)
except transform.TransformFailed as error:
    print(error)
    print("\nThe report was NOT rebuilt. It still shows the last numbers that passed every test.")

It is now Wednesday 2026-06-10 07:00:00 in Arden.


dbt stopped: 1 problem(s).
  FAILED accepted_values_orders_status__placed__packed__out_for_delivery__delivered__cancelled: Got 1 result, configured to fail if != 0
  Not built, because something before it failed: daily_orders

The report was NOT rebuilt. It still shows the last numbers that passed every test.


The program says which check failed and what was skipped. Somebody now **knows** the number would have been wrong.

In [8]:
table(con, "SELECT run_id, rows_landed, models_built, tests_passed, tests_failed FROM ops.load_runs ORDER BY run_id")

,run_id,rows_landed,models_built,tests_passed,tests_failed
0,2026-06-08_1,226,7,31,0
1,2026-06-09_1,102,7,31,0
2,2026-06-10_1,93,6,24,1


In [9]:
con.close()

## Step 5: The tests of the pipeline

There are two kinds of test in this milestone now.

| Kind | Where | It checks | Runs |
|---|---|---|---|
| Table tests | `dbt_project/` | The **data** of this morning | On every pipeline run |
| pytest | `tests/` | The **code**: that the pipeline is right, and that the table tests fail when they should | When the code changes |

The pytest suite runs dbt many times, so it takes over a minute.

In [10]:
!cd .. && uv run pytest -q --color=no 2>&1 | tail -n 2 | sed -E 's/ in [0-9.]+s.*//'

...............................                                          [100%]
31 passed


In [11]:
show_doc("LLD.md", "8. Checks on the pipeline itself (pytest)")

**From `docs/LLD.md`**

> ## 8. Checks on the pipeline itself (pytest)
> 
> | Check | Test |
> |---|---|
> | The report equals the answer key on every morning, 2 to 18 June | `test_the_report_equals_the_answer_key_on_every_morning` |
> | Running a morning twice changes nothing | `test_running_the_same_morning_twice_changes_nothing` |
> | An unknown status fails the run, and names the test | `test_an_unknown_status_fails_the_accepted_values_test` |
> | When a test fails, the report keeps its last good numbers | `test_when_a_test_fails_the_report_keeps_its_last_good_numbers` |
> | The incremental report equals a full rebuild | `test_the_incremental_report_equals_a_full_rebuild` |
> | An incremental run touches only recent days | `test_an_incremental_run_touches_only_recent_days` |
> | The clean tables can be rebuilt from raw alone | `test_the_clean_tables_can_be_rebuilt_from_raw_alone` |
> | Old data fails the freshness check | `test_fresh_data_passes_and_old_data_fails_the_freshness_check` |

## Step 6: What is still wrong

In [12]:
show_doc("LLD.md", "9. What is still wrong")

**From `docs/LLD.md`**

> ## 9. What is still wrong
> 
> Everything is right, and everything is checked. One thing is left.
> 
> | Morning | What must happen | Who does it |
> |---|---|---|
> | Every day at 07:00 | `uv run python -m daily_orders` | A person |
> | When the order system is down at 07:00 | Try again a little later | A person, if they notice |
> | When a week was missed | Run each missed day | A person, seven times |
> 
> **A person must remember to run it every morning.** Milestone 5 puts it on a schedule.

Everything in this notebook started with you running a cell. The pipeline is right, safe, checked, and completely dependent on a person.

## The words of milestone 4

| Word | In one or two sentences | You saw it in |
|---|---|---|
| **ELT** | Load the raw data, then transform it inside the warehouse | Notebook 1 |
| **Model** | One SELECT in a file. dbt builds a table or a view from it | Notebook 2 |
| **Type casting**, **cleaning** | Give each value its real type, and one spelling | Notebook 2 |
| **Deduplication** | Keep one row per key: the newest | Notebook 3 |
| **Join**, **enrichment** | Put two tables side by side where a key matches. Add columns from another table | Notebook 3 |
| **Row-by-row**, **set-based** | Visit one row at a time. Or say what you want from all the rows together | Notebook 4 |
| **Staging, core, mart** | Three layers: clean it, one row per real thing, what people read | Notebook 5 |
| **Incremental model** | A model that rebuilds only part of its table | Notebook 5 |
| **Table test** | A query that returns the rows that break a rule | Notebook 6 |
| **Freshness** | Is the newest loaded row recent enough? | Notebook 6 |

The project's [`GLOSSARY.md`](../../../GLOSSARY.md) keeps every word, with an interview line for each.

## What you now know

- Python extracts and loads. dbt transforms and tests. Together they are ELT.
- A failed table test stops the run, names itself, and protects the report.
- pytest checks the code. Table tests check this morning's data. You need both.

**You have finished milestone 4.** Open [`milestone_4_recap.html`](../milestone_4_recap.html) in a browser. Then open [`milestone_4_flow.html`](../milestone_4_flow.html) again.

> **Still wrong:** a person must remember to run it every morning. Milestone 5 puts it on a schedule.